In [ ]:
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN name IS NOT NULL THEN 1 ELSE 0 END) AS name_not_null,
    SUM(CASE WHEN rota_type IS NOT NULL THEN 1 ELSE 0 END) AS rota_type_not_null,
    SUM(CASE WHEN location IS NOT NULL THEN 1 ELSE 0 END) AS location_not_null,
    SUM(CASE WHEN id_branch IS NOT NULL THEN 1 ELSE 0 END) AS id_branch_not_null,
    SUM(CASE WHEN specialty IS NOT NULL THEN 1 ELSE 0 END) AS specialty_not_null,
    SUM(CASE WHEN treatment_code IS NOT NULL THEN 1 ELSE 0 END) AS treatment_code_not_null
FROM silver_sone_srrota

In [ ]:
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN Name IS NOT NULL THEN 1 ELSE 0 END) AS name_not_null,
    SUM(CASE WHEN RotaType IS NOT NULL THEN 1 ELSE 0 END) AS rota_type_not_null,
    SUM(CASE WHEN Location IS NOT NULL THEN 1 ELSE 0 END) AS location_not_null,
    SUM(CASE WHEN IDBranch IS NOT NULL THEN 1 ELSE 0 END) AS id_branch_not_null,
    SUM(CASE WHEN Specialty IS NOT NULL THEN 1 ELSE 0 END) AS specialty_not_null,
    SUM(CASE WHEN TreatmentCode IS NOT NULL THEN 1 ELSE 0 END) AS treatment_code_not_null
FROM bronze_sone_srrota

In [ ]:
SELECT
    RowIdentifier,
    Name,
    RotaType,
    Location,
    IDBranch,
    Specialty,
    TreatmentCode,
    FILEDATE,
    loaddatetime
FROM bronze_sone_srrota
WHERE RowIdentifier IN (
    SELECT id
    FROM silver_sone_srrota
    WHERE name IS NULL
    LIMIT 20
)
ORDER BY RowIdentifier, loaddatetime DESC

I reviewed the silver_sone_srrota create logic and checked the output against the bronze source data. The current logic selects the latest row per RowIdentifier using file date. However, for silver_sone_srrota, the latest row can sometimes be a null-heavy row, while an older row contains the actual populated values. Because of this, the current logic is bringing through nulls in silver. The logic therefore needs to be changed so that populated/useful rows are prioritised first, and the latest file date is applied only after that. Please confirm if I should proceed with updating the row selection logic in this way.

In [ ]:
I reviewed the silver_sone_srrota create logic and checked the output against the bronze source data. The current logic selects the latest row per RowIdentifier using file date. However, for silver_sone_srrota, the latest row can sometimes be a null-heavy row, while an older row contains the actual populated values. Because of this, the current logic is bringing through nulls in silver. The logic therefore needs to be changed so that populated/useful rows are prioritised first, and the latest file date is applied only after that. Please confirm if I should proceed with updating the row selection logic in this way.

In [ ]:
The output proves that the issue is not a missing source column. For the same RowIdentifier, both a null row and a populated row exist in bronze. The current logic is picking the wrong row because it uses latest file date only.

In [ ]:
Thanks Sean, I’ve checked this now.

`RemovedData` is present on the bronze table, and in the query output the null-heavy rows come through with `RemovedData = 1`, while the populated rows for the same `RowIdentifier` come through with `RemovedData = 0`.

So this does look like the null-heavy rows are delete markers rather than missing source data. Based on that, I think the correct approach would be to exclude `RemovedData = 1` rows from the ranking and return the latest non-deleted row in silver.

I’ve attached the query output showing this behaviour for the same `RowIdentifier`. Please let me know if you’re happy for me to proceed with that logic change.

: 

In [ ]:
Thanks Sean, I’ve checked this now.

`RemovedData` is present on the bronze table, and in the query output the null-heavy rows come through with `RemovedData = 1`, while the populated rows for the same `RowIdentifier` come through with `RemovedData = 0`.

So this does look like the null-heavy rows are delete markers rather than missing source data. Based on that, I think the correct approach would be to exclude `RemovedData = 1` rows from the ranking and return the latest non-deleted row in silver.

I’ve attached the query output showing this behaviour for the same `RowIdentifier`. Please let me know if you’re happy for me to proceed with that logic change.

In [ ]:
SELECT
    RemovedData,
    COUNT(*) AS row_count
FROM bronze_sone_srrota
WHERE RowIdentifier IN (
    SELECT id
    FROM silver_sone_srrota
    WHERE name IS NULL
)
GROUP BY RemovedData
ORDER BY RemovedData;

In [ ]:
SELECT
    RowIdentifier,
    RemovedData,
    COUNT(*) AS row_count
FROM bronze_sone_srrota
WHERE RowIdentifier IN (
    SELECT id
    FROM silver_sone_srrota
    WHERE name IS NULL
    LIMIT 20
)
GROUP BY RowIdentifier, RemovedData
ORDER BY RowIdentifier, RemovedData;

In [ ]:
Thanks Sean, I’ve checked this now.

`RemovedData` is present on the bronze table. I validated the affected `RowIdentifier` values and the query output shows that for the same `RowIdentifier`, both a `RemovedData = 0` row and a `RemovedData = 1` row exist in bronze.

This confirms that the null-heavy rows are delete-marker rows, and the current silver logic is picking those rows because it is ranking by latest file date only. Based on that, I think the correct approach would be to exclude `RemovedData = 1` rows from the ranking and return the latest non-deleted row in silver.

I’ve attached the query outputs showing this for the same `RowIdentifier`. Please let me know if you’re happy for me to proceed with that logic change.

In [ ]:
-- SILVER vs BRONZE comparison
SELECT
    s.id AS silver_id,
    s.name AS silver_name,
    b.RowIdentifier AS bronze_rowidentifier,
    b.Name AS bronze_name,
    b.RemovedData AS bronze_removed_data,
    b.FILEDATE AS bronze_filedate
FROM silver_sone_srrota s
LEFT JOIN bronze_sone_srrota b
    ON s.id = b.RowIdentifier
WHERE s.name IS NULL
ORDER BY s.id, b.loaddatetime DESC
LIMIT 50

In [ ]:
Thanks Sean, I’ve done some further checks on the bronze data.

What I found is:

1. For some affected `RowIdentifier` values, both `RemovedData = 0` and `RemovedData = 1` rows exist in bronze. In those cases, the null-heavy row appears to be the delete-marker row, so the current ranking logic can select the wrong row.
2. However, I also found some cases where the bronze row itself has `RemovedData = 0` and the source values are still null, so not all nulls are caused by the ranking logic alone.

So my understanding is that we do have a logic issue for the delete-marker cases, but updating the ranking to exclude `RemovedData = 1` rows will only fix that part of the problem. Some null rows may still remain because they are already null in bronze.

Please let me know if you are happy for me to:

* first update the ranking logic to exclude `RemovedData = 1` rows when selecting the latest active record
* then recheck the remaining null rows separately



In [ ]:
mpb_care_product AS (
    SELECT DISTINCT
    -- MPB: cprod_name = Therapy Type + Pathway
    CONCAT(tt.type, ' - ', apt.name) AS cprod_name,
    'MPB001' AS cprod_src_sys_inst_id,
    -- MPB: cprod_src_id = Therapy Type ID + Pathway ID
    CONCAT('MPB', CAST(a.therapy_type_id AS STRING), '-', CAST(a.appointment_type_id AS STRING)) AS cprod_src_id
    FROM silver_drj_appointments a
    LEFT JOIN silver_drj_therapy_types tt ON a.therapy_type_id = tt.id
    LEFT JOIN silver_drj_appointment_types apt ON a.appointment_type_id = apt.id
    WHERE a.therapy_type_id IS NOT NULL
      AND a.appointment_type_id IS NOT NULL
      AND tt.id IS NOT NULL
      AND apt.id IS NOT NULL
),